# Advanced Polynomial Regression: Feature Engineering and Model Selection

This notebook explores polynomial feature creation, degree selection, and preventing overfitting in polynomial models.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Generate non-linear synthetic data
np.random.seed(42)
X = np.linspace(-3, 3, 200).reshape(-1, 1)
y = 2 * X.squeeze()**3 - 5 * X.squeeze()**2 + 3 * X.squeeze() + np.random.randn(200) * 5

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Test different polynomial degrees
degrees = range(1, 11)
train_scores = []
test_scores = []
cv_scores = []

for degree in degrees:
    poly_features = PolynomialFeatures(degree=degree)
    X_train_poly = poly_features.fit_transform(X_train)
    X_test_poly = poly_features.transform(X_test)
    
    # Use Ridge to prevent overfitting
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train_poly, y_train)
    
    train_scores.append(ridge.score(X_train_poly, y_train))
    test_scores.append(ridge.score(X_test_poly, y_test))
    
    # Cross-validation score
    cv_score = cross_val_score(ridge, X_train_poly, y_train, cv=5).mean()
    cv_scores.append(cv_score)

# Plot model complexity vs performance
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(degrees, train_scores, 'o-', label='Train R²', linewidth=2)
plt.plot(degrees, test_scores, 's-', label='Test R²', linewidth=2)
plt.plot(degrees, cv_scores, '^-', label='CV R² (mean)', linewidth=2)
plt.xlabel('Polynomial Degree')
plt.ylabel('R² Score')
plt.title('Model Complexity vs Performance')
plt.legend()
plt.grid(True, alpha=0.3)

# Fit and visualize best model (degree 3)
poly_features = PolynomialFeatures(degree=3)
X_train_poly = poly_features.fit_transform(X_train)
X_test_poly = poly_features.transform(X_test)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_poly, y_train)

# Predictions for smooth curve
X_smooth = np.linspace(-3, 3, 300).reshape(-1, 1)
X_smooth_poly = poly_features.transform(X_smooth)
y_smooth = ridge.predict(X_smooth_poly)

plt.subplot(1, 2, 2)
plt.scatter(X_train, y_train, alpha=0.5, label='Training Data')
plt.scatter(X_test, y_test, alpha=0.5, label='Test Data')
plt.plot(X_smooth, y_smooth, 'r-', linewidth=2, label='Polynomial Fit (degree=3)')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Optimal Polynomial Fit')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best degree: 3")
print(f"Test R² score: {ridge.score(X_test_poly, y_test):.4f}")

# Polynomial Regression Analysis

Exploring polynomial features for non-linear relationship modeling.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import validation_curve
from sklearn.metrics import mean_squared_error

In [ ]:
# Generate non-linear dataset
np.random.seed(789)
X = np.linspace(0, 1, 50).reshape(-1, 1)
y = 1.5 * X.ravel() + 2 * X.ravel()**2 - 3 * X.ravel()**3 + 0.1 * np.random.randn(50)

# Polynomial degrees to test
degrees = np.arange(1, 8)
train_scores = []
val_scores = []

for degree in degrees:
    poly_reg = Pipeline([
        ('poly', PolynomialFeatures(degree=degree)),
        ('linear', LinearRegression())
    ])
    
    train_score, val_score = validation_curve(
        poly_reg, X, y, param_name='poly__degree', 
        param_range=[degree], cv=5, scoring='neg_mean_squared_error'
    )
    
    train_scores.append(-train_score.mean())
    val_scores.append(-val_score.mean())

print(f"Optimal degree: {degrees[np.argmin(val_scores)]}")